In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU — avoids DataParallel OOM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U "transformers>=4.50" "peft>=0.13" "datasets>=2.20" \
    "bitsandbytes>=0.43" accelerate sentencepiece rouge_score sacrebleu evaluate "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 76.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.4 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = "hf_xxx"
login(HF_TOKEN)

In [3]:
import torch, gc
try: del model, base
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

In [4]:

TRANSLATED_CSV   = "/kaggle/input/datasets/mathiaskabango/translated-csv/srh_translated.csv"
EXTRA_NATIVE_CSV = "/kaggle/input/datasets/mathiaskabango/train1-df/train_df.csv"
NATIVE_KIN_CSV   = "/kaggle/input/datasets/mathiaskabango/kinyarw/kinyarwanda.csv"

BASE_MODEL   = "google/gemma-3-1b-it"
ADAPTER_DIR  = "/kaggle/input/models/mathiaskabango/model/pytorch/default/1/gemma3-1b-srh-lora/checkpoint-800"

COMPARE_TO_BASE = True

SEED = 42
MAX_NEW_TOKENS = 200

## Reconstruct the held-out eval set (same seed/logic as training)


In [5]:
import pandas as pd, re, numpy as np

def reshape_translated(df):
    rows = []
    cols = {"Eng": ("question_en","answer_en"), "Kin": ("question_kin","answer_kin"),
            "Swa": ("question_swa","answer_swa")}
    for _, r in df.iterrows():
        for lang, (qc, ac) in cols.items():
            q, a = r.get(qc), r.get(ac)
            if isinstance(q, str) and isinstance(a, str) and q.strip() and a.strip():
                rows.append({"lang": lang, "question": q.strip(), "answer": a.strip()})
    return pd.DataFrame(rows)

translated = reshape_translated(pd.read_csv(TRANSLATED_CSV))

native = pd.read_csv(EXTRA_NATIVE_CSV).rename(columns={"instruction":"question","response":"answer"})
native["lang"] = native["lang"].replace({"Eng_Uga":"Eng","Eng_Ken":"Eng","Kin_Rwa":"Kin",
                                         "Swa_Ken":"Swa","Lug_Uga":"Lug"})
native = native[["lang","question","answer"]]

kin_extra = pd.read_csv(NATIVE_KIN_CSV)
if "instruction" in kin_extra.columns:
    kin_extra = kin_extra.rename(columns={"instruction":"question","response":"answer"})
kin_extra = kin_extra[["question","answer"]].copy(); kin_extra["lang"] = "Kin"

data = pd.concat([translated, native, kin_extra[["lang","question","answer"]]], ignore_index=True)
data = data[data["lang"].isin(["Eng","Kin","Swa"])].reset_index(drop=True)  # Luganda excluded from eval

def clean(t):
    t = str(t)
    t = re.sub(r"as of my last (knowledge update|update)", "", t, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", t).strip()
data["question"] = data["question"].map(clean)
data["answer"]   = data["answer"].map(clean)
data = data[(data["question"]!="") & (data["answer"]!="")]
data = data.drop_duplicates(subset=["lang","question","answer"]).reset_index(drop=True)

print("Full pool:")
print(data["lang"].value_counts())

Full pool:
lang
Swa    11228
Kin     9753
Eng     9423
Name: count, dtype: int64


In [6]:
# Same held-out logic as training: 15% held out per language, BEFORE oversampling.
eval_parts = []
for lang in ["Eng", "Kin", "Swa"]:
    sub = data[data["lang"] == lang]
    eval_parts.append(sub.sample(frac=0.15, random_state=SEED))
eval_set = pd.concat(eval_parts, ignore_index=True).reset_index(drop=True)

print("Held-out eval set (never trained on):")
print(eval_set["lang"].value_counts())
print(f"Total: {len(eval_set)}")

# Optional: cap for a faster run while iterating. Set to None for the full set.
PER_LANG_EVAL_CAP = 40  # ⚙️ set to None to evaluate everything
if PER_LANG_EVAL_CAP:
    eval_set = eval_set.groupby("lang", group_keys=False).apply(
        lambda g: g.sample(min(PER_LANG_EVAL_CAP, len(g)), random_state=SEED)
    ).reset_index(drop=True)
    print(f"\nCapped to {PER_LANG_EVAL_CAP}/language for this run:")
    print(eval_set["lang"].value_counts())

Held-out eval set (never trained on):
lang
Swa    1684
Kin    1463
Eng    1413
Name: count, dtype: int64
Total: 4560

Capped to 40/language for this run:
lang
Eng    40
Kin    40
Swa    40
Name: count, dtype: int64


/tmp/ipykernel_58/2609729515.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  eval_set = eval_set.groupby("lang", group_keys=False).apply(


## Load the fine-tuned model (4-bit + adapter)

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb,
    torch_dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)   # model object first, adapter path second
model.config.use_cache = True
model.eval()

# Stop-token fix: use BOTH <eos> and <end_of_turn>, not just the auto-aligned <eos>.
eot_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
eos_id = tokenizer.eos_token_id
stop_ids = [t for t in {eot_id, eos_id} if t is not None]
print("Stop token ids:", stop_ids)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Stop token ids: [1, 106]


In [8]:
@torch.no_grad()
def ask(model_, tok_, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = tok_.apply_chat_template([{"role":"user","content":question}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = tok_(prompt, return_tensors="pt").to(model_.device)
    out = model_.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        repetition_penalty=1.3, no_repeat_ngram_size=3,
        eos_token_id=stop_ids, pad_token_id=tok_.pad_token_id or tok_.eos_token_id,
    )
    return tok_.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

## Generate predictions (fine-tuned model)

In [9]:
preds = []
for i, row in eval_set.iterrows():
    preds.append(ask(model, tokenizer, row["question"]))
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(eval_set)}")

eval_set["prediction"] = preds
print("Done generating with fine-tuned model.")

  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
Done generating with fine-tuned model.




Same questions, same stop tokens, no adapter — for a clean before/after comparison.

In [10]:
if COMPARE_TO_BASE:
    import gc
    del model, base
    gc.collect(); torch.cuda.empty_cache()

    base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_only = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb,
        torch_dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="eager",
    )
    base_only.config.use_cache = True
    base_only.eval()

    base_eot = base_tok.convert_tokens_to_ids("<end_of_turn>")
    base_stop_ids_list = [t for t in {base_eot, base_tok.eos_token_id} if t is not None]

    @torch.no_grad()
    def ask_base(question, max_new_tokens=MAX_NEW_TOKENS):
        prompt = base_tok.apply_chat_template([{"role":"user","content":question}],
                                              tokenize=False, add_generation_prompt=True)
        inputs = base_tok(prompt, return_tensors="pt").to(base_only.device)
        out = base_only.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=3,
            eos_token_id=base_stop_ids_list,
            pad_token_id=base_tok.pad_token_id or base_tok.eos_token_id,
        )
        return base_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    base_preds = []
    for i, row in eval_set.iterrows():
        base_preds.append(ask_base(row["question"]))
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(eval_set)}")
    eval_set["prediction_base"] = base_preds
    print("Done generating with base (untuned) model.")
else:
    print("Skipped base-model comparison (COMPARE_TO_BASE=False).")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
Done generating with base (untuned) model.


## Score: ROUGE + chrF, per language and overall

In [11]:
import evaluate
rouge = evaluate.load("rouge")
chrf  = evaluate.load("chrf")

def score(preds, refs):
    r = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    c = chrf.compute(predictions=preds, references=[[x] for x in refs])
    return {"ROUGE-1": round(r["rouge1"]*100,2), "ROUGE-2": round(r["rouge2"]*100,2),
            "ROUGE-L": round(r["rougeL"]*100,2), "chrF": round(c["score"],2)}

rows = []
for lang in ["Eng", "Kin", "Swa"]:
    sub = eval_set[eval_set["lang"] == lang]
    if len(sub) == 0: continue
    s = score(sub["prediction"].tolist(), sub["answer"].tolist())
    s.update({"lang": lang, "model": "fine-tuned", "n": len(sub)})
    rows.append(s)

overall = score(eval_set["prediction"].tolist(), eval_set["answer"].tolist())
overall.update({"lang": "ALL", "model": "fine-tuned", "n": len(eval_set)})
rows.append(overall)

if COMPARE_TO_BASE:
    for lang in ["Eng", "Kin", "Swa"]:
        sub = eval_set[eval_set["lang"] == lang]
        if len(sub) == 0: continue
        s = score(sub["prediction_base"].tolist(), sub["answer"].tolist())
        s.update({"lang": lang, "model": "base (untuned)", "n": len(sub)})
        rows.append(s)
    overall_base = score(eval_set["prediction_base"].tolist(), eval_set["answer"].tolist())
    overall_base.update({"lang": "ALL", "model": "base (untuned)", "n": len(eval_set)})
    rows.append(overall_base)

results = pd.DataFrame(rows)[["model","lang","n","ROUGE-1","ROUGE-2","ROUGE-L","chrF"]]
results = results.sort_values(["lang","model"]).reset_index(drop=True)
print(results.to_string(index=False))
results.to_csv("/kaggle/working/eval_scores.csv", index=False)

         model lang   n  ROUGE-1  ROUGE-2  ROUGE-L  chrF
base (untuned)  ALL 120    12.79     1.30     7.19 21.66
    fine-tuned  ALL 120    21.28     4.35    11.96 30.50
base (untuned)  Eng  40    23.65     3.11    11.43 31.53
    fine-tuned  Eng  40    26.76     5.97    14.30 34.70
base (untuned)  Kin  40     3.55     0.16     3.12 15.49
    fine-tuned  Kin  40    15.59     3.11     8.88 26.56
base (untuned)  Swa  40    11.21     0.64     7.12 19.34
    fine-tuned  Swa  40    21.55     3.82    12.61 30.99


## Read the actual outputs, not just the scores

This is the important step. Aggregate metrics cannot tell fluent-but-wrong apart
from correct-but-differently-phrased — especially for Kinyarwanda.

In [12]:
cols = ["lang","question","answer","prediction"]
if COMPARE_TO_BASE: cols.append("prediction_base")

eval_set[cols].to_csv("/kaggle/working/eval_predictions_full.csv", index=False)

for lang in ["Eng", "Kin", "Swa"]:
    sub = eval_set[eval_set["lang"] == lang].head(3)
    for _, row in sub.iterrows():
        print(f"=== {lang} ===")
        print("Q        :", row["question"][:150])
        print("REF      :", row["answer"][:200])
        print("FINE-TUNE:", row["prediction"][:200])
        if COMPARE_TO_BASE:
            print("BASE     :", row["prediction_base"][:200])
        print("-"*70)

=== Eng ===
Q        : Define AIDS?
REF      : HIV is Human Immunodeficiency Virus. It is a virus that attacks and weakens the body's immune system. AIDS, Acquired Immunodeficiency Syndrome is the late stage of HIV infection where the body's immun
FINE-TUNE: Acquired Immunodeficiency Syndrome (AIDS) ni ugonjwa wa kiafya unaopitishwa kingono. Inaweza kutibiwa kwa matibabu yanayofaa na kuondokana kabla ya kukua hadi inawezekane kupunguza hatari yako ya kuwa
BASE     : Okay, here's a comprehensive definition of HIV/AIDS:

**HIV/AIDS stands for Human Immunodeficiency Virus. It’s not an illness itself; it’s the chronic infection that causes Acquired Immune Deficiency 
----------------------------------------------------------------------
=== Eng ===
Q        : Does Syphilis run in families?
REF      : Syphilis is an STI, it does not run in families, however, it can be transmitted from mother to child (vertical transmission) during pregnancy and child birth.
FINE-TUNE: Syphilis can be transm

## Kinyarwanda review bundle for your native speaker

Same two-tier structure as before: this is what actually validates whether the
model is *correct*, not just fluent.

In [15]:
kin_review = eval_set[eval_set["lang"] == "Kin"][cols]
kin_review.to_csv("/kaggle/working/kin_native_review.csv", index=False)
print(f"Saved {len(kin_review)} Kinyarwanda rows -> kin_native_review.csv for native review")

print("\nAll outputs saved to /kaggle/working/:")
print("  eval_scores.csv              (ROUGE/chrF summary table)")
print("  eval_predictions_full.csv    (every prediction, all languages)")
print("  kin_native_review.csv        (Kinyarwanda only, for native-speaker review)")

Saved 40 Kinyarwanda rows -> kin_native_review.csv for native review

All outputs saved to /kaggle/working/:
  eval_scores.csv              (ROUGE/chrF summary table)
  eval_predictions_full.csv    (every prediction, all languages)
  kin_native_review.csv        (Kinyarwanda only, for native-speaker review)
